In [13]:
import os
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image


# ==========================================
# 1. MODEL CONFIGURATION
# ==========================================

import os

BASE_DIR = os.getcwd()

MODEL_PATH = os.path.abspath(
    os.path.join(
        BASE_DIR,
        "model",
        "plant_disease_recog_model_pwp.keras"
    )
)

IMG_SIZE = (224, 224)

print("Model path:", MODEL_PATH)
print("Model exists:", os.path.exists(MODEL_PATH))

# ==========================================
# 2. LOAD MODEL
# ==========================================

def load_disease_model():
    """Load the trained plant disease detection model."""

    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(
            f"Model file not found: {MODEL_PATH}"
        )

    model = load_model(MODEL_PATH)

    print("Plant disease model loaded successfully!")

    return model


# ============================================================
# PREDICT PLANT DISEASE
# ============================================================

def predict_disease(model, image_path, class_names):

    import os
    import numpy as np
    from tensorflow.keras.preprocessing import image

    if not os.path.isfile(image_path):
        raise FileNotFoundError(
            f"Image not found: {image_path}"
        )

    # Get image size directly from the model
    input_shape = model.input_shape

    img_height = input_shape[1]
    img_width = input_shape[2]

    # Load and resize image
    img = image.load_img(
        image_path,
        target_size=(img_height, img_width)
    )

    # Convert image to NumPy array
    img_array = image.img_to_array(img)

    # Add batch dimension
    img_array = np.expand_dims(img_array, axis=0)

    # IMPORTANT:
    # Apply the same normalization used during model training.
    # If training used rescale=1./255, uncomment:
    # img_array = img_array / 255.0

    # Predict
    predictions = model.predict(img_array, verbose=0)[0]

    predicted_index = int(np.argmax(predictions))
    confidence = float(predictions[predicted_index]) * 100

    if predicted_index >= len(class_names):
        raise ValueError(
            "The number of class names does not match the model output."
        )

    predicted_class = class_names[predicted_index]

    return {
        "plant_disease": predicted_class,
        "confidence": round(confidence, 2)
    }


# ==========================================
# 4. MAIN EXECUTION
# ==========================================

if __name__ == "__main__":

    # Load the model only once
    model = load_disease_model()

    # Replace these examples with your exact
    # training class names and their original order.
    # ============================================================
# 3. PLANT DISEASE CLASS NAMES
# ============================================================

CLASS_NAMES = [
    "Apple_scab",
    "Apple_black_rot",
    "Apple_cedar_apple_rust",
    "Apple_healthy",
    "Background_without_leaves",
    "Blueberry_healthy",
    "Cherry_powdery_mildew",
    "Cherry_healthy",
    "Corn_gray_leaf_spot",
    "Corn_common_rust",
    "Corn_northern_leaf_blight",
    "Corn_healthy",
    "Grape_black_rot",
    "Grape_black_measles",
    "Grape_leaf_blight",
    "Grape_healthy",
    "Orange_haunglongbing",
    "Peach_bacterial_spot",
    "Peach_healthy",
    "Pepper_bacterial_spot",
    "Pepper_healthy",
    "Potato_early_blight",
    "Potato_healthy",
    "Potato_late_blight",
    "Raspberry_healthy",
    "Soybean_healthy",
    "Squash_powdery_mildew",
    "Strawberry_healthy",
    "Strawberry_leaf_scorch",
    "Tomato_bacterial_spot",
    "Tomato_early_blight",
    "Tomato_healthy",
    "Tomato_late_blight",
    "Tomato_leaf_mold",
    "Tomato_septoria_leaf_spot",
    "Tomato_spider_mites_two-spotted_spider_mite",
    "Tomato_target_spot",
    "Tomato_mosaic_virus",
    "Tomato_yellow_leaf_curl_virus"
]

print("Total classes:", len(CLASS_NAMES))

assert len(CLASS_NAMES) == 39, "Expected exactly 38 classes."

if not CLASS_NAMES:
    raise ValueError(
        "Add the 38 class names in the exact order "
        "used during model training."
    )

    

Model path: c:\Users\KIIT0001\OneDrive\Desktop\major_1\major-1\model\plant_disease_recog_model_pwp.keras
Model exists: True
Plant disease model loaded successfully!
Total classes: 39


In [14]:
# ============================================================
# EXPORT CLASS NAMES
# ============================================================

import json
import os

CLASS_NAMES_PATH = os.path.join(
    "backend",
    "class_names.json"
)

os.makedirs(
    "backend",
    exist_ok=True
)

with open(CLASS_NAMES_PATH, "w") as file:

    json.dump(
        CLASS_NAMES,
        file,
        indent=4
    )

print("Class names exported successfully!")

print("Total classes:", len(CLASS_NAMES))

Class names exported successfully!
Total classes: 39


In [15]:
# ============================================================
# CHECK MODEL INPUT AND OUTPUT SHAPES
# ============================================================

print("Model input shape:", model.input_shape)
print("Model output shape:", model.output_shape)
print("Image size currently configured:", IMG_SIZE)

Model input shape: (None, 160, 160, 3)
Model output shape: (None, 39)
Image size currently configured: (224, 224)


In [16]:
# ============================================================
# CHECK IMAGE PREPROCESSING
# ============================================================

from tensorflow.keras.preprocessing import image
import numpy as np

# Use the exact input size expected by your trained model
IMG_SIZE = (160, 160)

img = image.load_img(
    image_path,
    target_size=IMG_SIZE
)

img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)

print("Image batch shape:", img_array.shape)
print("Image data type:", img_array.dtype)
print("Image pixel range:", img_array.min(), "to", img_array.max())

Image batch shape: (1, 160, 160, 3)
Image data type: float32
Image pixel range: 0.0 to 255.0


In [17]:
# ============================================================
# TEST PLANT DISEASE PREDICTION
# ============================================================

image_path = os.path.abspath("../test1.jpg")

result = predict_disease(
    model,
    image_path,
    CLASS_NAMES
)

print("Prediction Result")
print("-----------------")
print("Disease:", result["plant_disease"])
print("Confidence:", f'{result["confidence"]}%')

Prediction Result
-----------------
Disease: Apple_scab
Confidence: 100.0%


In [18]:
# ============================================================
# TEST PLANT DISEASE PREDICTION
# ============================================================

image_path = os.path.abspath("../test2.webp")

result = predict_disease(
    model,
    image_path,
    CLASS_NAMES
)

print("Prediction Result")
print("-----------------")
print("Disease:", result["plant_disease"])
print("Confidence:", f'{result["confidence"]}%')

Prediction Result
-----------------
Disease: Background_without_leaves
Confidence: 100.0%


In [19]:
# ============================================================
# TEST PLANT DISEASE PREDICTION
# ============================================================

image_path = os.path.abspath("../test3.jpg")

result = predict_disease(
    model,
    image_path,
    CLASS_NAMES
)

print("Prediction Result")
print("-----------------")
print("Disease:", result["plant_disease"])
print("Confidence:", f'{result["confidence"]}%')

Prediction Result
-----------------
Disease: Background_without_leaves
Confidence: 100.0%


In [20]:
# ============================================================
# TEST PLANT DISEASE PREDICTION
# ============================================================

image_path = os.path.abspath("../test4.jpg")

result = predict_disease(
    model,
    image_path,
    CLASS_NAMES
)

print("Prediction Result")
print("-----------------")
print("Disease:", result["plant_disease"])
print("Confidence:", f'{result["confidence"]}%')

Prediction Result
-----------------
Disease: Corn_healthy
Confidence: 100.0%


In [22]:
print("Model output shape:", model.output_shape)

print("\nFinal model layers:")

for layer in model.layers[-5:]:
    print(layer.name, layer.output.shape)

Model output shape: (None, 39)

Final model layers:
input_layer_6 (None, 160, 160, 3)


AttributeError: 'list' object has no attribute 'shape'

In [23]:
print("Total dataset classes:", len(CLASS_NAMES))

for i, name in enumerate(CLASS_NAMES):
    print(i, name)

Total dataset classes: 39
0 Apple_scab
1 Apple_black_rot
2 Apple_cedar_apple_rust
3 Apple_healthy
4 Background_without_leaves
5 Blueberry_healthy
6 Cherry_powdery_mildew
7 Cherry_healthy
8 Corn_gray_leaf_spot
9 Corn_common_rust
10 Corn_northern_leaf_blight
11 Corn_healthy
12 Grape_black_rot
13 Grape_black_measles
14 Grape_leaf_blight
15 Grape_healthy
16 Orange_haunglongbing
17 Peach_bacterial_spot
18 Peach_healthy
19 Pepper_bacterial_spot
20 Pepper_healthy
21 Potato_early_blight
22 Potato_healthy
23 Potato_late_blight
24 Raspberry_healthy
25 Soybean_healthy
26 Squash_powdery_mildew
27 Strawberry_healthy
28 Strawberry_leaf_scorch
29 Tomato_bacterial_spot
30 Tomato_early_blight
31 Tomato_healthy
32 Tomato_late_blight
33 Tomato_leaf_mold
34 Tomato_septoria_leaf_spot
35 Tomato_spider_mites_two-spotted_spider_mite
36 Tomato_target_spot
37 Tomato_mosaic_virus
38 Tomato_yellow_leaf_curl_virus
